# Amazon S3 Tables PoC — Multi-Engine Access & Batch Ingestion

This notebook validates Spark/PyIceberg access to S3 Tables and performs batch data ingestion.

**Run this notebook from:**
- Amazon SageMaker AI Notebook (JupyterLab)
- Amazon SageMaker Studio
- Any local IDE with Python 3.9+ (VS Code, PyCharm, etc.)

**Prerequisites:**
- Deploy the CloudFormation stack first (see README)
- Create the Glue federated catalog and namespace (see README Phase 1)
- AWS credentials configured (IAM role on SageMaker, or `~/.aws/credentials` locally)
- `pip install pyiceberg[s3,pyarrow] boto3 pyarrow pandas`

## Setup

In [ ]:
# Install dependencies (run once)
!pip install -q "pyiceberg[s3,pyarrow]" boto3 pyarrow pandas

In [ ]:
import boto3
import json

# Configuration — update these to match your deployment
AWS_REGION = "us-east-1"  # <-- your deployment region
STACK_NAME = "s3-tables-poc"

# Auto-discover from CloudFormation outputs
cfn = boto3.client("cloudformation", region_name=AWS_REGION)
outputs = {o["OutputKey"]: o["OutputValue"] 
           for o in cfn.describe_stacks(StackName=STACK_NAME)["Stacks"][0]["Outputs"]}

TABLE_BUCKET_ARN = outputs["TableBucketARN"]
TABLE_BUCKET_NAME = outputs["TableBucketName"]
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

print(f"Region:       {AWS_REGION}")
print(f"Table Bucket: {TABLE_BUCKET_ARN}")
print(f"Account:      {ACCOUNT_ID}")

## 1. Connect to S3 Tables via PyIceberg REST Catalog

In [ ]:
from pyiceberg.catalog import load_catalog

catalog = load_catalog(
    "s3tables",
    type="rest",
    warehouse=TABLE_BUCKET_ARN,
    uri=f"https://s3tables.{AWS_REGION}.amazonaws.com/iceberg",
    **{
        "rest.sigv4-enabled": "true",
        "rest.signing-name": "s3tables",
        "rest.signing-region": AWS_REGION,
    }
)

# List namespaces
print("Namespaces:", catalog.list_namespaces())

## 2. Multi-Engine Validation — Read Athena-Created Table

In [ ]:
# List tables in poc_data namespace
print("Tables:", catalog.list_tables("poc_data"))

In [ ]:
# Read the 'customers' table created by Athena in Phase 1
customers = catalog.load_table("poc_data.customers")
df = customers.scan().to_pandas()
print(f"Rows from Athena-created table: {len(df)}")
df.sort_values("id")

## 3. Multi-Engine Validation — Write from PyIceberg

In [ ]:
import pyarrow as pa
from datetime import datetime

# Write new rows from PyIceberg
new_rows = pa.table({
    "id": [10, 11],
    "name": ["PyIceberg User 1", "PyIceberg User 2"],
    "email": ["pyiceberg1@example.com", "pyiceberg2@example.com"],
    "created_at": [datetime.now(), datetime.now()],
})

customers.append(new_rows)
print("Appended 2 rows from PyIceberg")

# Verify
df = customers.scan().to_pandas()
print(f"Total rows now: {len(df)}")
df.sort_values("id")

## 4. Batch Ingestion — Create Events Table & Load 50K Records

In [ ]:
from pyiceberg.schema import Schema
from pyiceberg.types import (
    StringType, IntegerType, DoubleType, TimestampType, NestedField
)
from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.transforms import DayTransform

# Define schema
events_schema = Schema(
    NestedField(1, "event_id", StringType(), required=True),
    NestedField(2, "event_type", StringType(), required=True),
    NestedField(3, "user_id", IntegerType(), required=True),
    NestedField(4, "amount", DoubleType(), required=False),
    NestedField(5, "event_time", TimestampType(), required=True),
    NestedField(6, "region", StringType(), required=True),
)

# Partition by day(event_time)
partition_spec = PartitionSpec(
    PartitionField(source_id=5, field_id=1000, transform=DayTransform(), name="event_day")
)

# Create table
try:
    events_table = catalog.create_table(
        "poc_data.events",
        schema=events_schema,
        partition_spec=partition_spec,
    )
    print("Created events table")
except Exception as e:
    if "already exists" in str(e).lower():
        events_table = catalog.load_table("poc_data.events")
        print("Events table already exists, loaded it")
    else:
        raise

In [ ]:
import random
import uuid
from datetime import datetime, timedelta

EVENT_TYPES = ["purchase", "click", "view", "refund", "signup"]
REGIONS = ["us-east-1", "eu-west-1", "ap-southeast-1", "us-west-2"]
NUM_RECORDS = 50_000
BATCH_SIZE = 10_000

print(f"Generating and loading {NUM_RECORDS:,} records in batches of {BATCH_SIZE:,}...")

for batch_start in range(0, NUM_RECORDS, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, NUM_RECORDS)
    n = batch_end - batch_start
    
    batch = pa.table({
        "event_id": [str(uuid.uuid4()) for _ in range(n)],
        "event_type": [random.choice(EVENT_TYPES) for _ in range(n)],
        "user_id": [random.randint(1, 1000) for _ in range(n)],
        "amount": [round(random.uniform(1.0, 500.0), 2) for _ in range(n)],
        "event_time": [
            datetime.now() - timedelta(days=random.randint(0, 6), 
                                       hours=random.randint(0, 23),
                                       minutes=random.randint(0, 59))
            for _ in range(n)
        ],
        "region": [random.choice(REGIONS) for _ in range(n)],
    })
    
    events_table.append(batch)
    print(f"  Loaded batch {batch_start+1:,}–{batch_end:,}")

print(f"\nDone. Total records loaded: {NUM_RECORDS:,}")

In [ ]:
# Verify the load
events_table = catalog.load_table("poc_data.events")
df = events_table.scan().to_pandas()
print(f"Total rows in events table: {len(df):,}")
print(f"\nBy event_type:")
print(df.groupby("event_type").agg({"event_id": "count", "amount": "sum"}).rename(columns={"event_id": "count"}))
print(f"\nBy region:")
print(df.groupby("region")["event_id"].count())

## 5. Verify Cross-Engine — Query from Athena

Run this from your terminal (not the notebook) to confirm Athena can read the PyIceberg-written data:

```bash
aws athena start-query-execution \
  --query-string "SELECT event_type, count(*) as cnt, round(sum(amount),2) as total_amount \
    FROM s3tablescatalog.\"${TableBucketName}\".poc_data.events \
    GROUP BY event_type ORDER BY cnt DESC" \
  --work-group $AthenaWorkgroupName --region $AWS_REGION
```

## 6. Table Metadata Inspection

In [ ]:
# Inspect table metadata
events_table = catalog.load_table("poc_data.events")

print(f"Schema: {events_table.schema()}")
print(f"Partition spec: {events_table.spec()}")
print(f"Current snapshot: {events_table.current_snapshot()}")
print(f"\nSnapshot history:")
for snap in events_table.history():
    print(f"  {snap}")

In [ ]:
# Check maintenance status via S3 Tables API
s3tables = boto3.client("s3tables", region_name=AWS_REGION)

maintenance = s3tables.get_table_maintenance_configuration(
    tableBucketARN=TABLE_BUCKET_ARN,
    namespace="poc_data",
    name="events"
)
print("Maintenance configuration:")
print(json.dumps(maintenance["configuration"], indent=2, default=str))

job_status = s3tables.get_table_maintenance_job_status(
    tableBucketARN=TABLE_BUCKET_ARN,
    namespace="poc_data",
    name="events"
)
print("\nMaintenance job status:")
print(json.dumps(job_status["status"], indent=2, default=str))

---

**Next steps**: Return to the README for Phase 2 (Firehose streaming), Phase 3 (CloudWatch observability), and Phase 4 (administration).